<a href="https://colab.research.google.com/github/AmnonElias/DeepLearningProject/blob/main/CombiningDataSets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
import zipfile
import shutil
import yaml
from google.colab import drive

# 1. חיבור ל-Google Drive
drive.mount('/content/drive')

# 2. הגדרות ומיפוי מחלקות סופי (האינדקסים החדשים שלך)
# 0: dog, 1: cat, 2: snake, 3: mouse, 4: rat, 5: cockroach

# הגדרת תיקיית היעד המאוחדת (שתיווצר ב-Colab)
unified_dataset_path = '/content/drive/My Drive/DeepLearning/unified_dataset'
splits = ['train', 'valid', 'test']

# יצירת מבנה התיקיות של YOLO
for split in splits:
    os.makedirs(os.path.join(unified_dataset_path, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(unified_dataset_path, split, 'labels'), exist_ok=True)

# 3. הגדרת מקורות המידע והמיפויים
# עליך לעדכן את נתיבי ה-ZIP ולהגדיר איזה אינדקס ישן הופך לאיזה אינדקס חדש
datasets_config = [
    {
        # לדוגמה: דאטהסט שמכיל רק עכברים. ב-Roboflow העכבר קיבל אינדקס 0, אבל אצלנו הוא יהיה 3
        'zip_path': '/content/drive/My Drive/DeepLearning/DataSets/Mice.v8i.yolov12.zip',
        'mapping': {0: 3}
    },
    {
        'zip_path': '/content/drive/My Drive/DeepLearning/DataSets/rats.v2i.yolov12.zip',
        'mapping': {0: 4}
    },
    {
        'zip_path': '/content/drive/My Drive/DeepLearning/DataSets/Snakes.v1i.yolov12.zip',
        'mapping': {0: 2}
    },
    {
        'zip_path': '/content/drive/My Drive/DeepLearning/DataSets/cats.v1i.yolov12_1.zip',
        'mapping': {0: 1}
    },
    {
        'zip_path': '/content/drive/My Drive/DeepLearning/DataSets/Cats.v1i.yolov12.zip',
        'mapping': {0: 1}
    },
    {
        'zip_path': '/content/drive/My Drive/DeepLearning/DataSets/cockroach.v1i.yolov12.zip',
        'mapping': {0: 5}
    },
    {
        'zip_path': '/content/drive/My Drive/DeepLearning/DataSets/cockroach.v2i.yolov12.zip',
        'mapping': {0: 5}
    },
    {
        'zip_path': '/content/drive/My Drive/DeepLearning/DataSets/dogs.v1i.yolov12.zip',
        'mapping': {0: 0}
    },
    {
        'zip_path': '/content/drive/My Drive/DeepLearning/DataSets/dogs.v3i.yolov12.zip',
        'mapping': {0: 0}
    },
    {
        'zip_path': '/content/drive/My Drive/DeepLearning/DataSets/Rats.v4i.yolov12.zip',
        'mapping': {0: 4}
    },
    {
        'zip_path': '/content/drive/My Drive/DeepLearning/DataSets/snakes.v14i.yolov12.zip',
        'mapping': {0: 2}
    }

]

# 4. לולאת חילוץ, המרה ואיחוד
extract_dir = '/content/drive/My Drive/DeepLearning/temp_extract'

for ds_idx, ds in enumerate(datasets_config):
    print(f"מעבד את דאטהסט מספר {ds_idx + 1}...")
    zip_path = ds['zip_path']
    mapping = ds['mapping']

    # חילוץ ה-ZIP לתיקייה זמנית
    if not os.path.exists(zip_path):
        print(f"שגיאה: הקובץ {zip_path} לא נמצא. מדלג.")
        continue

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    # מעבר על חלוקות הדאטה (train, valid, test)
    for split in splits:
        # Roboflow לרוב מחלק לתיקיות משנה של images ו-labels
        src_images = os.path.join(extract_dir, split, 'images')
        src_labels = os.path.join(extract_dir, split, 'labels')

        dest_images = os.path.join(unified_dataset_path, split, 'images')
        dest_labels = os.path.join(unified_dataset_path, split, 'labels')

        if not os.path.exists(src_images):
            continue

        # מעבר על כל התמונות בתיקייה
        for filename in os.listdir(src_images):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                # כדי למנוע התנגשויות שמות בין דאטהסטים שונים, נוסיף קידומת לשם הקובץ
                new_filename = f"ds{ds_idx}_{filename}"
                label_filename = os.path.splitext(filename)[0] + '.txt'
                new_label_filename = os.path.splitext(new_filename)[0] + '.txt'

                # העתקת התמונה
                shutil.copy(os.path.join(src_images, filename), os.path.join(dest_images, new_filename))

                # טיפול בקובץ התיוג (.txt)
                src_label_path = os.path.join(src_labels, label_filename)
                dest_label_path = os.path.join(dest_labels, new_label_filename)

                if os.path.exists(src_label_path):
                    with open(src_label_path, 'r') as f_in, open(dest_label_path, 'w') as f_out:
                        for line in f_in:
                            parts = line.strip().split()
                            if len(parts) >= 5: # מוודא שהשורה תקינה (class, x, y, w, h)
                                old_class = int(parts[0])
                                # מעדכן את המחלקה רק אם היא מופיעה במיפוי
                                if old_class in mapping:
                                    new_class = mapping[old_class]
                                    parts[0] = str(new_class)
                                    f_out.write(' '.join(parts) + '\n')

    # מחיקת התיקייה הזמנית כדי לפנות מקום לפני הדאטהסט הבא
    if os.path.exists(extract_dir):
        shutil.rmtree(extract_dir)

# 5. יצירת קובץ data.yaml מאוחד חדש
yaml_content = {
    'train': f"{unified_dataset_path}/train/images",
    'val': f"{unified_dataset_path}/valid/images",
    'test': f"{unified_dataset_path}/test/images",
    'nc': 6,
    'names': ['dog', 'cat', 'snake', 'mouse', 'rat', 'cockroach']
}

with open(os.path.join(unified_dataset_path, 'data.yaml'), 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print(f"האיחוד הושלם בהצלחה! הדאטהסט המאוחד מוכן בנתיב: {unified_dataset_path}")
print("ניתן כעת להתחיל לאמן את מודל ה-YOLO עליו.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
מעבד את דאטהסט מספר 1...
מעבד את דאטהסט מספר 2...
מעבד את דאטהסט מספר 3...
מעבד את דאטהסט מספר 4...
מעבד את דאטהסט מספר 5...
מעבד את דאטהסט מספר 6...
מעבד את דאטהסט מספר 7...
מעבד את דאטהסט מספר 8...
מעבד את דאטהסט מספר 9...
מעבד את דאטהסט מספר 10...
מעבד את דאטהסט מספר 11...
האיחוד הושלם בהצלחה! הדאטהסט המאוחד מוכן בנתיב: /content/drive/My Drive/DeepLearning/unified_dataset
ניתן כעת להתחיל לאמן את מודל ה-YOLO עליו.
